# 02 · Profiling y reglas de calidad

**Objetivo:** medir las seis dimensiones y separar errores críticos de advertencias.

**Modalidad:** material para explicar en clase. Los ejemplos se ejecutan únicamente en memoria, con datos ficticios. No conectan a una base, no crean archivos ni ejecutan una ingesta. Los bloques SQL son ilustrativos y no se envían a ningún servidor.

**Prerrequisitos:** Python con pandas. Cada notebook es independiente: no requiere scripts `.py` ni haber ejecutado otra notebook.


In [ ]:
# Importar conexión centralizada desde el módulo
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO, USUARIO

# Obtener los engines
engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas correctamente.')
print(f'Servidor: {SERVIDOR}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')


## De la unidad III al contrato de datos

| Dimensión | Pregunta | Ejemplo |
|---|---|---|
| Completitud | ¿Falta algo obligatorio? | ID presente |
| Unicidad | ¿Se repite una entidad? | PK duplicada |
| Integridad | ¿Existe la referencia? | Producto conocido |
| Consistencia | ¿Se respetan tipos y relaciones? | Decimal y fecha válidos |
| Vigencia | ¿La extracción llega a tiempo? | SLA de ingesta |
| Exactitud | ¿Coincide con la realidad? | Comparación con fuente autorizada |

Un precio con formato correcto no demuestra exactitud. La fecha antigua de una
venta no prueba atraso de la ingesta. `LastEditedWhen` sin zona no es UTC por
definición.


In [ ]:
import pandas as pd

datos = pd.DataFrame([
    {"id": 1, "producto": 10, "cantidad": 2, "total": "24.20"},
    {"id": 2, "producto": 999, "cantidad": 1, "total": "ERROR"},
    {"id": 2, "producto": 20, "cantidad": None, "total": "6.05"},
])
perfil = pd.DataFrame({
    "tipo_observado": datos.dtypes.astype(str),
    "nulos": datos.isna().sum(),
    "valores_distintos": datos.nunique(dropna=True),
})
display(datos)
display(perfil)


In [ ]:
from decimal import Decimal, InvalidOperation

def decimal_valido(valor):
    try:
        numero = Decimal(str(valor))
        return numero.is_finite() and numero == numero.quantize(Decimal("0.01"))
    except (InvalidOperation, ValueError):
        return False

fallas = pd.DataFrame({
    "completitud_cantidad": datos["cantidad"].isna(),
    "unicidad_id": datos["id"].duplicated(keep=False),
    "integridad_producto": ~datos["producto"].isin([10, 20]),
    "consistencia_total": ~datos["total"].map(decimal_valido),
})
display(fallas)
print("Filas rechazadas:", int(fallas.any(axis=1).sum()))
print("Incumplimientos:", int(fallas.sum().sum()))


## Contrato de ejemplo

Reglas versionadas `ventas-clase-1`: cada una tiene identificador, dimensión,
severidad, umbral, responsable y acción. Para identificadores críticos,
duplicados y referencias proponemos tolerancia cero.

El ejemplo tiene **dos filas rechazadas y cinco incumplimientos**. No sumar
incumplimientos para obtener cantidad de filas.

`UnitPrice` puede ser NULL en WWI; `TaxRate` admite tres decimales. No declarar
obligatoria una columna sin conocer el esquema y el uso de negocio.


In [ ]:
# Ejemplo local de contrato con la API de Great Expectations 0.18.22.
# Ejecutar esta celda requiere la versión opcional indicada en requirements.txt.
import great_expectations as gx

validador = gx.from_pandas(datos)
resultado = validador.expect_column_values_to_be_unique("id")
print("¿ID único?:", resultado.success)
print("Valores inesperados:", resultado.result.get("unexpected_count"))


## Actividad de discusión

¿Un outlier debe ir automáticamente a cuarentena? **No necesariamente:** puede ser legítimo; se necesita una regla de negocio, severidad y revisión.